# Demo GSTL (risiko tabrakan dari video dashcam) di Google Colab

Notebook ini menjalankan `demo.py` dari repo `gstl_replication` pada **video Anda sendiri**: skor risiko per frame, grafik, dan video overlay. Semua hasil disimpan ke Google Drive Anda di `Research/gstl_replication/hasil/<video>_<waktu>_<detektor>/` (termasuk `run_info.json`: video, detektor, commit kode, waktu).

**Sebelum mulai**
1. Menu *Runtime → Change runtime type → T4 GPU*.
2. Repo privat? Simpan token GitHub (fine-grained, hanya baca repo ini) di *Secrets* (ikon kunci) dengan nama `GITHUB_TOKEN`, lalu aktifkan *Notebook access*. Jangan menulis token di sel.
3. Taruh `latest.pth` (969 MB) dan `vgg16-397923af.pth` (528 MB) di Google Drive Anda pada `Research/gstl_replication/` (sel 4 membaca dari sana). Hanya memakai YOLO (sel 6b)? Sel 4 boleh dilewati.

**Keterbatasan** (jangan diklaim lebih): model dilatih hanya pada CCD 1280x720; video lain di luar distribusi latih; skor bukan risiko terkalibrasi; detektor tanpa kelas motor. Notebook ini belum teruji di semua versi Colab.

In [ ]:
# 1. Periksa GPU dan versi
import sys, torch, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (demo tetap jalan di CPU, jauh lebih lambat)")

In [ ]:
# 2. Ambil kode dari GitHub (token dibaca dari Secrets bila ada; tidak pernah dicetak)
import os, subprocess
REPO = "taufiqmus/gstl_replication"
if not os.path.isdir("/content/gstl_replication"):
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = None
    url = ("https://%s@github.com/%s.git" % (tok, REPO)) if tok else ("https://github.com/%s.git" % REPO)
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, "/content/gstl_replication"], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("clone gagal (repo privat tanpa token, atau token salah): " + r.stderr.replace(tok or "<none>", "***"))
    if tok:  # hapus token dari konfigurasi git di mesin Colab
        subprocess.run(["git", "-C", "/content/gstl_replication", "remote", "set-url", "origin", "https://github.com/%s.git" % REPO])
%cd /content/gstl_replication
!git log --oneline -1

In [ ]:
# 3. Pasang dependensi demo (torch dan numpy bawaan Colab TIDAK diganti)
!pip -q install "opencv-python-headless" "Pillow" "matplotlib"
import cv2; print("opencv", cv2.__version__)

In [ ]:
# 3b. Hubungkan Google Drive: bobot dibaca dari sini, semua hasil ditulis ke sini
import os, shutil, json, subprocess, time
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Research/gstl_replication"   # bobot (latest.pth, vgg16-397923af.pth)
HASIL_DIR = DRIVE_DIR + "/hasil"                                  # satu subfolder per proses
os.makedirs(HASIL_DIR, exist_ok=True)

def simpan_ke_drive(lokal, nama, **info):
    """Salin folder keluaran lokal ke Drive (menulis mp4 langsung ke Drive rawan gagal), tambah run_info.json."""
    commit = subprocess.run(["git", "-C", "/content/gstl_replication", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
    info.update(waktu=time.strftime("%Y-%m-%d %H:%M:%S"), commit=commit)
    json.dump(info, open(lokal + "/run_info.json", "w"), indent=1, ensure_ascii=False)
    tujuan = HASIL_DIR + "/" + nama
    shutil.copytree(lokal, tujuan, dirs_exist_ok=True)
    print("tersimpan di Drive:", tujuan)
    return tujuan

In [ ]:
# 4. Bobot dari Google Drive Anda (Research/gstl_replication): detektor Cascade R-CNN dan VGG16
DET = "/content/gstl_replication/weights/detector/latest.pth"   # bobot Cascade (milik penulis UString, 969 MB)
os.makedirs(os.path.dirname(DET), exist_ok=True)
if not os.path.exists(DET):
    shutil.copy(DRIVE_DIR + "/latest.pth", DET)

# VGG16 ImageNet ke cache torch hub supaya torchvision tidak mengunduh ulang (528 MB)
HUB = os.path.expanduser("~/.cache/torch/hub/checkpoints")
os.makedirs(HUB, exist_ok=True)
VGG = HUB + "/vgg16-397923af.pth"
if not os.path.exists(VGG):
    shutil.copy(DRIVE_DIR + "/vgg16-397923af.pth", VGG)
import hashlib
h = hashlib.sha256(open(VGG, "rb").read()).hexdigest()
assert h.startswith("397923af"), "VGG16 rusak/berbeda: " + h[:8]

# Periksa hash detektor SEBELUM dipakai (berkas dimuat dengan unpickle penuh)
!python tools/check_setup.py --detector "{DET}" --hash-detector


In [ ]:
# 5. Unggah video (mp4/avi/mov). Atau isi VIDEO dengan path di Drive/Colab dan lewati unggahan.
VIDEO = ""
if not VIDEO:
    from google.colab import files
    up = files.upload()
    VIDEO = "/content/" + next(iter(up))
    os.replace(next(iter(up)), VIDEO)
print("video:", VIDEO)
RUN = os.path.splitext(os.path.basename(VIDEO))[0] + "_" + time.strftime("%Y%m%d-%H%M%S")
print("nama proses:", RUN)


In [ ]:
# 6. Jalankan demo (detektor Cascade). MAX_WINDOWS=0 memproses seluruh video; isi mis. 4 untuk uji cepat (4 jendela x 50 frame).
MAX_WINDOWS = 0
LOKAL = "/content/hasil_lokal/" + RUN + "_cascade"
!python -W ignore demo.py --video "{VIDEO}" --detector "{DET}" --out "{LOKAL}" --viz full --uncertainty --html --threshold 0.5 --width 960 --max-windows {MAX_WINDOWS}
OUT = simpan_ke_drive(LOKAL, RUN + "_cascade", video=os.path.basename(VIDEO), detektor="cascade_rcnn", max_windows=MAX_WINDOWS, threshold=0.5)

### 6b. (Opsional) Detektor YOLOv8 — tidak perlu `latest.pth`
Alternatif yang jauh lebih ringan: bobot `yolov8s.pt` (±22 MB) diunduh otomatis, dan sel 4 (bobot Cascade, 969 MB) boleh dilewati. Lebih cepat dan mengenali motor/pengendara, **tetapi bukan pipeline CCD asli**: GSTL dilatih dengan fitur dan kotak Cascade, jadi skornya bisa berbeda dan kesetaraannya belum diuji. `ultralytics` dan bobot YOLO berlisensi **AGPL-3.0**.

Setelah sel ini, jalankan ulang sel 7 untuk menampilkan hasil YOLO.

In [ ]:
# 6b. Jalankan demo dengan YOLOv8s (ambang deteksi 0,25 bawaan)
!pip -q install ultralytics
from ultralytics import YOLO
YOLO("yolov8s.pt")   # mengunduh yolov8s.pt ke folder kerja saat ini
YOLO_W = os.path.abspath("yolov8s.pt")
MAX_WINDOWS = globals().get("MAX_WINDOWS", 0)
LOKAL = "/content/hasil_lokal/" + RUN + "_yolo"
!python -W ignore demo.py --video "{VIDEO}" --det-model yolov8 --detector "{YOLO_W}" --out "{LOKAL}" --viz full --uncertainty --html --threshold 0.5 --width 960 --max-windows {MAX_WINDOWS}
OUT = simpan_ke_drive(LOKAL, RUN + "_yolo", video=os.path.basename(VIDEO), detektor="yolov8s", max_windows=MAX_WINDOWS, threshold=0.5)

In [ ]:
# 7. Tampilkan grafik skor dan video overlay
from IPython.display import Image, display, HTML
from base64 import b64encode
print(sorted(os.listdir(OUT)))
display(Image(OUT + "/skor.png"))
if os.path.exists(OUT + "/overlay.mp4"):
    data = b64encode(open(OUT + "/overlay.mp4", "rb").read()).decode()
    display(HTML('<video width="720" controls src="data:video/mp4;base64,%s"></video>' % data))

In [ ]:
# 8. Ringkasan: semua hasil sudah ada di Drive (tidak hilang saat sesi berakhir)
print("folder hasil:", HASIL_DIR)
for d in sorted(os.listdir(HASIL_DIR)):
    n = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(HASIL_DIR + "/" + d) for f in fs)
    print("  %-60s %7.1f MB" % (d, n / 1e6))

## Catatan
- Skor dihitung per jendela 50 frame (keadaan GRU di-reset), jadi skor rendah pada ±1 detik pertama tiap jendela itu **artefak**, bukan perubahan risiko.
- Bila `latest.pth` tidak tersedia, `demo.py` punya `--det-model tv_fasterrcnn` (bobot COCO, diunduh otomatis), tetapi itu bukan pipeline CCD asli.
- Video overlay panjang bisa besar; untuk uji cepat pakai `MAX_WINDOWS = 4`.